# Proyecto 01 · Retail Intelligence — A1: Auditoría y Preparación de Datos
Tópicos Especiales para la Gestión de Datos (UCAB).

**Qué hace este notebook:** carga los 6 CSV, mide su calidad (completitud, unicidad, consistencia, oportunidad), detecta outliers e inconsistencias, aplica las decisiones de limpieza documentadas y genera los **datasets maestros** que usa todo el equipo en A2, A3 y A4.

**Cómo usarlo en Colab:** sube los 6 CSV al panel de archivos (o a una carpeta de Drive) y ajusta `RUTA`. Ejecuta todo con *Entorno de ejecución → Ejecutar todas*.

In [ ]:
import pandas as pd
import numpy as np

RUTA = './'          # carpeta donde están los CSV
SALIDA = './salida/' # carpeta donde se guardan los datasets limpios
import os; os.makedirs(SALIDA, exist_ok=True)

INICIO, FIN = pd.Timestamp('2023-01-01'), pd.Timestamp('2024-12-31')  # ventana del caso (24 meses)
VENTANA_ATRIBUCION = 30   # días para atribuir una compra a una campaña
VENTANA_DEVOLUCION = 30   # política de devolución observada en los datos

clientes = pd.read_csv(RUTA+'clientes.csv', parse_dates=['fecha_registro'])
trx      = pd.read_csv(RUTA+'transacciones.csv', parse_dates=['fecha'])
productos= pd.read_csv(RUTA+'productos.csv')
dev      = pd.read_csv(RUTA+'devoluciones.csv', parse_dates=['fecha_devolucion'])
crm      = pd.read_csv(RUTA+'interacciones_crm.csv', parse_dates=['fecha'])
cam      = pd.read_csv(RUTA+'campanas_marketing.csv', parse_dates=['fecha_envio'])

datasets = {'clientes':clientes,'transacciones':trx,'productos':productos,
            'devoluciones':dev,'interacciones_crm':crm,'campanas_marketing':cam}
pks = {'clientes':'cliente_id','transacciones':'transaccion_id','productos':'producto_id',
       'devoluciones':'devolucion_id','interacciones_crm':'interaccion_id','campanas_marketing':'campana_id'}
pd.DataFrame({k:[len(v), v.shape[1]] for k,v in datasets.items()}, index=['filas','columnas']).T

## 1. Perfil general: nulos, nulos "ocultos" y dominios
Un nulo no siempre es `NaN`: valores como `No especificado` son **nulos ocultos** y bajan la completitud real del campo.

In [ ]:
NULOS_OCULTOS = ['No especificado','N/A','NA','', 'desconocido']
perfil = []
for n, df in datasets.items():
    for col in df.columns:
        nul = df[col].isna().sum()
        ocu = df[col].astype(str).isin(NULOS_OCULTOS).sum()
        perfil.append([n, col, str(df[col].dtype), nul, ocu, round(100*(1-(nul+ocu)/len(df)),2), df[col].nunique()])
perfil = pd.DataFrame(perfil, columns=['dataset','campo','tipo','nulos','nulos_ocultos','completitud_%','valores_distintos'])
perfil[perfil['completitud_%']<100]

In [ ]:
# Dominios de las variables categóricas (comparar contra lo que dice el enunciado)
for n, df in datasets.items():
    for col in df.select_dtypes(exclude=['number','datetime']).columns:
        if df[col].nunique() < 15:
            print(f'{n}.{col}:', df[col].value_counts().to_dict())

## 2. Unicidad: duplicados exactos, de clave y lógicos
- **Exactos:** filas idénticas.
- **De clave:** el ID primario repetido.
- **Lógicos:** mismo evento con distinto ID (ej. mismo cliente, misma campaña, mismo día).

In [ ]:
claves_logicas = {
 'transacciones':['cliente_id','fecha','producto_id','cantidad','precio_unitario'],
 'devoluciones':['transaccion_id'],
 'interacciones_crm':['cliente_id','tipo','fecha','canal'],
 'campanas_marketing':['cliente_id','tipo_campana','fecha_envio'],
 'clientes':['cliente_id'], 'productos':['nombre']}
uni = []
for n, df in datasets.items():
    uni.append([n, df.duplicated().sum(), df[pks[n]].duplicated().sum(), df.duplicated(claves_logicas[n]).sum(),
                round(100*df[pks[n]].nunique()/len(df),2)])
pd.DataFrame(uni, columns=['dataset','dup_exactos','dup_clave','dup_logicos','unicidad_%'])

In [ ]:
# El único duplicado lógico: misma campaña al mismo cliente el mismo día, con distinto descuento
cam[cam.duplicated(claves_logicas['campanas_marketing'], keep=False)]

In [ ]:
# Integridad del catálogo: el enunciado dice 320 productos
faltan = sorted(set(f'PRD{i:04d}' for i in range(1,321)) - set(productos.producto_id))
print('Productos en catálogo:', len(productos), '| IDs faltantes:', faltan)
print('Productos sin ventas:', (~productos.producto_id.isin(trx.producto_id)).sum())

## 3. Consistencia: reglas de dominio, referenciales, de negocio y temporales
Cada regla devuelve True/False por fila. La **consistencia del dataset** es el % de filas que cumple todas sus reglas.

In [ ]:
primera_compra = trx.groupby('cliente_id').fecha.min()
reg = clientes.set_index('cliente_id').fecha_registro
tp = trx.merge(productos, on='producto_id', how='left')
dm = dev.merge(trx[['transaccion_id','cliente_id','fecha','monto_neto']], on='transaccion_id', how='left', suffixes=('','_trx'))

# Validación de conversiones: ¿el cliente compró después del envío?
def compra_en_ventana(df_cam, dias):
    x = df_cam[['campana_id','cliente_id','fecha_envio']].merge(trx[['cliente_id','fecha']], on='cliente_id', how='left')
    x['ok'] = (x.fecha >= x.fecha_envio) & (x.fecha <= x.fecha_envio + pd.Timedelta(days=dias))
    return x.groupby('campana_id').ok.any()
cam_ok = compra_en_ventana(cam, VENTANA_ATRIBUCION)

reglas = {
 'clientes': {
   'edad en [18,100]': clientes.edad.between(18,100),
   'genero en dominio': clientes.genero.isin(['F','M','No especificado']),
   'registro dentro de la ventana': clientes.fecha_registro.between(INICIO,FIN),
   'registro <= primera compra': ~(clientes.cliente_id.map(primera_compra) < clientes.fecha_registro)},
 'transacciones': {
   'cliente existe (FK)': trx.cliente_id.isin(clientes.cliente_id),
   'producto existe (FK)': trx.producto_id.isin(productos.producto_id),
   'cantidad > 0': trx.cantidad > 0,
   'descuento en [0,1)': trx.descuento.between(0, 0.9999),
   'precio_unitario = precio_lista': (tp.precio_unitario - tp.precio_lista).abs() < 0.01,
   'monto_neto = cant*precio*(1-desc)': ((trx.cantidad*trx.precio_unitario*(1-trx.descuento)).round(2) - trx.monto_neto).abs() <= 0.05,
   'fecha dentro de la ventana': trx.fecha.between(INICIO,FIN)},
 'productos': {
   'costo < precio_lista': productos.costo < productos.precio_lista,
   'margen_pct recalculado coincide': (((productos.precio_lista-productos.costo)/productos.precio_lista*100).round(1) - productos.margen_bruto_pct).abs() <= 0.2},
 'devoluciones': {
   'transacción existe (FK)': dm.fecha.notna(),
   'cliente coincide con la transacción': dm.cliente_id == dm.cliente_id_trx,
   'devolución >= fecha de compra': dm.fecha_devolucion >= dm.fecha,
   'monto_devuelto <= monto_neto': dm.monto_devuelto <= dm.monto_neto + 0.01,
   'devuelto en <= 30 días': (dm.fecha_devolucion - dm.fecha).dt.days <= VENTANA_DEVOLUCION},
 'interacciones_crm': {
   'cliente existe (FK)': crm.cliente_id.isin(clientes.cliente_id),
   'tiempo_respuesta > 0': crm.tiempo_respuesta_hrs > 0,
   'interacción >= registro': crm.fecha >= crm.cliente_id.map(reg)},
 'campanas_marketing': {
   'cliente existe (FK)': cam.cliente_id.isin(clientes.cliente_id),
   'convertido implica abierto': ~((cam.convertido==1) & (cam.abierto==0)),
   'envío >= registro': cam.fecha_envio >= cam.cliente_id.map(reg),
   f'convertida con compra en {VENTANA_ATRIBUCION} días': ~((cam.convertido==1) & ~cam.campana_id.map(cam_ok))},
}
filas = []
for n, rs in reglas.items():
    for r, s in rs.items():
        filas.append([n, r, int((~s).sum()), round(100*s.mean(),2)])
detalle_reglas = pd.DataFrame(filas, columns=['dataset','regla','filas_que_fallan','cumplimiento_%'])
detalle_reglas

### 3.1 Hallazgo crítico: `fecha_registro` no es confiable
Si 3 de cada 4 clientes compraron antes de "registrarse", el campo no mide el alta real del cliente. Probablemente registra otra cosa (ej. alta en el programa digital o última actualización de ficha).
**Decisión:** no eliminar registros. Crear `fecha_alta_efectiva = min(fecha_registro, primera_compra)` y usarla para antigüedad, cohortes y CLV.

In [ ]:
cl = clientes.copy()
cl['primera_compra'] = cl.cliente_id.map(primera_compra)
inc = cl.primera_compra < cl.fecha_registro
print(f'Clientes con compra antes del registro: {inc.sum()} de {cl.primera_compra.notna().sum()} con compras ({100*inc.mean():.1f}% del total)')
print('Días de adelanto (mediana):', (cl.fecha_registro - cl.primera_compra)[inc].dt.days.median())

### 3.2 Hallazgo crítico: la conversión de campañas no siempre tiene compra detrás
El enunciado pide revisar "campañas con fechas posteriores a la compra". Probamos si cada campaña marcada `convertido=1` tiene una compra del mismo cliente dentro de N días después del envío.

In [ ]:
conv = cam[cam.convertido==1]
noconv = cam[cam.convertido==0]
sens = []
for dias in [7, 30, 60, 90]:
    ok_c = compra_en_ventana(conv, dias); ok_n = compra_en_ventana(noconv, dias)
    sens.append([dias, int(ok_c.sum()), round(100*ok_c.mean(),1), round(100*ok_n.mean(),1)])
pd.DataFrame(sens, columns=['ventana_dias','convertidas_validadas','%_convertidas_con_compra','%_NO_convertidas_con_compra'])

**Lectura:** con 30 días, solo ~22% de las "convertidas" tienen compra posterior, casi lo mismo que las NO convertidas (~20%). El flag `convertido` es una **atribución declarada** por la plataforma, no trazable a una transacción.
**Decisión:** A4 usa el flag declarado (es el que define la tasa del caso: 22% → 11%) pero reporta también la conversión validada como prueba de robustez y como limitación del dato.

## 4. Outliers
Se usa el rango intercuartílico (IQR). Con k=1.5 se marcan atípicos leves y con k=3 extremos. Un outlier **no se elimina** si tiene explicación de negocio.

In [ ]:
def iqr_flag(s, k):
    q1, q3 = s.quantile([.25,.75]); return (s < q1-k*(q3-q1)) | (s > q3+k*(q3-q1))
out = []
for n, df, col in [('transacciones',trx,'monto_neto'),('transacciones',trx,'cantidad'),('transacciones',trx,'descuento'),
                   ('devoluciones',dev,'monto_devuelto'),('interacciones_crm',crm,'tiempo_respuesta_hrs'),
                   ('clientes',clientes,'edad'),('productos',productos,'costo')]:
    s = df[col]
    out.append([n, col, s.min(), s.median(), s.max(), int(iqr_flag(s,1.5).sum()), int(iqr_flag(s,3).sum())])
pd.DataFrame(out, columns=['dataset','campo','min','mediana','max','atipicos_k1.5','extremos_k3'])

In [ ]:
# ¿Los montos extremos se explican por la categoría?
print(tp[iqr_flag(tp.monto_neto,3)].categoria.value_counts())
# Ventas por debajo del costo (descuento mayor que el margen)
bajo = tp[tp.precio_unitario*(1-tp.descuento) < tp.costo]
print('\nVentas bajo costo:', len(bajo)); print(bajo.categoria.value_counts())
# Tiempos de respuesta mayores a una semana
print('\nInteracciones con respuesta > 168 h:', (crm.tiempo_respuesta_hrs>168).sum())

## 5. Oportunidad (timeliness)
Mide si el dato está disponible a tiempo y cubre el periodo completo. Tres ángulos:
1. **Cobertura:** % de registros dentro de la ventana de 24 meses.
2. **Rezago del proceso:** días entre el evento y su registro (devoluciones) o tiempo de atención (CRM).
3. **Censura a la derecha:** registros recientes cuyo seguimiento aún no terminó (ej. compras de diciembre 2024 que todavía podrían devolverse).

In [ ]:
fechas = {'clientes':clientes.fecha_registro,'transacciones':trx.fecha,'devoluciones':dev.fecha_devolucion,
          'interacciones_crm':crm.fecha,'campanas_marketing':cam.fecha_envio}
op = []
for n, f in fechas.items():
    op.append([n, f.min().date(), f.max().date(), round(100*f.between(INICIO,FIN).mean(),2)])
op.append(['productos','—','—',None])
oport = pd.DataFrame(op, columns=['dataset','fecha_min','fecha_max','cobertura_%'])
lag = (dm.fecha_devolucion - dm.fecha).dt.days
censuradas = (trx.fecha > FIN - pd.Timedelta(days=VENTANA_DEVOLUCION)).mean()
print(oport.to_string(index=False))
print(f'\nDevoluciones: rezago mediano {lag.median():.0f} días (máx {lag.max()}).')
print(f'CRM: {100*(crm.tiempo_respuesta_hrs<=24).mean():.1f}% respondidas en <= 24 h; mediana {crm.tiempo_respuesta_hrs.median():.1f} h.')
print(f'Transacciones con ventana de devolución incompleta (dic-2024): {100*censuradas:.1f}%')

## 6. Tabla resumen de calidad por dataset

In [ ]:
resumen = []
for n, df in datasets.items():
    celdas = df.size
    huecos = df.isna().sum().sum() + df.astype(str).isin(NULOS_OCULTOS).sum().sum()
    cons = pd.concat(reglas[n], axis=1).all(axis=1).mean() if n in reglas else np.nan
    cob = oport.set_index('dataset').loc[n,'cobertura_%']
    resumen.append([n, len(df), round(100*(1-huecos/celdas),2), round(100*df[pks[n]].nunique()/len(df),2),
                    round(100*cons,2), cob])
calidad = pd.DataFrame(resumen, columns=['dataset','filas','completitud_%','unicidad_%','consistencia_%','oportunidad_cobertura_%'])
calidad.to_csv(SALIDA+'metricas_calidad.csv', index=False)
detalle_reglas.to_csv(SALIDA+'detalle_reglas_consistencia.csv', index=False)
perfil.to_csv(SALIDA+'perfil_campos.csv', index=False)
calidad

## 7. Limpieza y transformación → datasets maestros
Decisiones (justificadas en el documento del equipo):
1. No se elimina ningún registro: ninguna fila es imposible; los problemas se **marcan con flags** para que cada análisis decida.
2. `fecha_alta_efectiva` reemplaza a `fecha_registro` para antigüedad y cohortes.
3. Se calculan costo, margen bruto y flags de devolución a nivel de transacción (base de A2 y A3).
4. Las campañas reciben `convertido_validado_30d` y el monto comprado en la ventana (base del ROI en A4).
5. Outliers de monto se conservan (son Electrónica y Hogar); tiempos de CRM > 168 h se marcan y los promedios usan la mediana.

In [ ]:
# --- Transacciones maestras (tabla de hechos enriquecida) ---
f = trx.merge(productos[['producto_id','categoria','subcategoria','costo','margen_bruto_pct']], on='producto_id', how='left')
f = f.merge(clientes[['cliente_id','segmento','edad','genero','ciudad','canal_adquisicion']], on='cliente_id', how='left')
f = f.merge(dev[['transaccion_id','motivo','monto_devuelto','fecha_devolucion']], on='transaccion_id', how='left')
f['anio'] = f.fecha.dt.year; f['mes'] = f.fecha.dt.month
f['anio_mes'] = f.fecha.dt.to_period('M').astype(str); f['trimestre'] = f.fecha.dt.to_period('Q').astype(str)
f['ingreso_bruto'] = (f.cantidad*f.precio_unitario).round(2)
f['monto_descuento'] = (f.ingreso_bruto - f.monto_neto).round(2)
f['costo_total'] = (f.cantidad*f.costo).round(2)
f['margen_bruto'] = (f.monto_neto - f.costo_total).round(2)
f['margen_pct_real'] = (100*f.margen_bruto/f.monto_neto).round(2)
f['venta_bajo_costo'] = (f.margen_bruto < 0).astype(int)
f['tiene_descuento'] = (f.descuento > 0).astype(int)
f['devuelto'] = f.monto_devuelto.notna().astype(int)
f['monto_devuelto'] = f.monto_devuelto.fillna(0)
f['monto_neto_final'] = (f.monto_neto - f.monto_devuelto).round(2)
f['margen_neto_devol'] = np.where(f.devuelto==1, 0, f.margen_bruto)   # supuesto: devolución anula la venta y su margen
f['ventana_devol_incompleta'] = (f.fecha > FIN - pd.Timedelta(days=VENTANA_DEVOLUCION)).astype(int)
f['outlier_monto'] = iqr_flag(f.monto_neto, 3).astype(int)
f = f.rename(columns={'motivo':'motivo_devolucion'})
f.to_csv(SALIDA+'transacciones_maestra.csv', index=False)

# --- Clientes maestros (dimensión con métricas RFM base) ---
agg = f.groupby('cliente_id').agg(primera_compra=('fecha','min'), ultima_compra=('fecha','max'),
        n_compras=('transaccion_id','count'), monto_total=('monto_neto','sum'), margen_total=('margen_bruto','sum'),
        n_devoluciones=('devuelto','sum'), monto_devuelto_total=('monto_devuelto','sum'))
cm = clientes.merge(agg, on='cliente_id', how='left')
cm['registro_inconsistente'] = (cm.primera_compra < cm.fecha_registro).astype(int)
cm['fecha_alta_efectiva'] = cm[['fecha_registro','primera_compra']].min(axis=1)
cm['recencia_dias'] = (FIN - cm.ultima_compra).dt.days
cm['antiguedad_dias'] = (FIN - cm.fecha_alta_efectiva).dt.days
cm['sin_compras'] = cm.n_compras.isna().astype(int)
cm[['n_compras','monto_total','margen_total','n_devoluciones','monto_devuelto_total']] = cm[['n_compras','monto_total','margen_total','n_devoluciones','monto_devuelto_total']].fillna(0)
q = crm.groupby('cliente_id').agg(n_interacciones=('interaccion_id','count'),
        n_quejas=('tipo', lambda s: (s=='queja').sum()))
q2 = crm[(crm.tipo=='queja') & (crm.resolucion=='sin_resolver')].groupby('cliente_id').size().rename('n_quejas_sin_resolver')
cm = cm.merge(q, on='cliente_id', how='left').merge(q2, on='cliente_id', how='left')
cm[['n_interacciones','n_quejas','n_quejas_sin_resolver']] = cm[['n_interacciones','n_quejas','n_quejas_sin_resolver']].fillna(0).astype(int)
cm.to_csv(SALIDA+'clientes_maestra.csv', index=False)

# --- CRM limpio ---
cr = crm.merge(clientes[['cliente_id','segmento']], on='cliente_id', how='left')
cr['outlier_tiempo'] = (cr.tiempo_respuesta_hrs > 168).astype(int)
cr['antes_de_registro'] = (cr.fecha < cr.cliente_id.map(reg)).astype(int)
cr.to_csv(SALIDA+'crm_limpio.csv', index=False)

# --- Campañas limpias (con validación y monto atribuible) ---
x = cam[['campana_id','cliente_id','fecha_envio']].merge(trx[['cliente_id','fecha','monto_neto']], on='cliente_id', how='left')
x = x[(x.fecha >= x.fecha_envio) & (x.fecha <= x.fecha_envio + pd.Timedelta(days=VENTANA_ATRIBUCION))]
att = x.groupby('campana_id').agg(monto_compra_30d=('monto_neto','sum'), primera_compra_post=('fecha','min'))
ca = cam.merge(clientes[['cliente_id','segmento']], on='cliente_id', how='left').merge(att, on='campana_id', how='left')
ca['monto_compra_30d'] = ca.monto_compra_30d.fillna(0).round(2)
ca['dias_a_compra'] = (ca.primera_compra_post - ca.fecha_envio).dt.days
ca['compra_30d'] = (ca.monto_compra_30d > 0).astype(int)
ca['convertido_validado_30d'] = ((ca.convertido==1) & (ca.compra_30d==1)).astype(int)
ca['antes_de_registro'] = (ca.fecha_envio < ca.cliente_id.map(reg)).astype(int)
ca['dup_logico'] = ca.duplicated(claves_logicas['campanas_marketing'], keep=False).astype(int)
ca.to_csv(SALIDA+'campanas_limpia.csv', index=False)

# --- Productos y dimensión fecha ---
productos.to_csv(SALIDA+'productos_limpio.csv', index=False)
dim_fecha = pd.DataFrame({'fecha': pd.date_range(INICIO, FIN)})
dim_fecha['anio']=dim_fecha.fecha.dt.year; dim_fecha['mes']=dim_fecha.fecha.dt.month
dim_fecha['anio_mes']=dim_fecha.fecha.dt.to_period('M').astype(str); dim_fecha['trimestre']=dim_fecha.fecha.dt.to_period('Q').astype(str)
dim_fecha['dia_semana']=dim_fecha.fecha.dt.day_name(); dim_fecha['fin_de_semana']=(dim_fecha.fecha.dt.dayofweek>=5).astype(int)
dim_fecha.to_csv(SALIDA+'dim_fecha.csv', index=False)
print({n: len(pd.read_csv(SALIDA+n)) for n in sorted(os.listdir(SALIDA)) if n.endswith('.csv')})

## 8. Cifras base para el equipo (sanity check)
Todos deben obtener estos mismos totales al cargar los datasets maestros. Si no coinciden, algo se cargó mal.

In [ ]:
print(f"Ingreso neto total:  {f.monto_neto.sum():,.2f}")
print(f"Margen bruto total:  {f.margen_bruto.sum():,.2f}  ({100*f.margen_bruto.sum()/f.monto_neto.sum():.1f}%)")
print(f"Monto devuelto:      {f.monto_devuelto.sum():,.2f}  ({100*f.devuelto.mean():.1f}% de transacciones)")
print(f"Ticket promedio:     {f.monto_neto.mean():,.2f}")
print(f"Conversión campañas: declarada {100*ca.convertido.mean():.1f}% | validada 30d {100*ca.convertido_validado_30d.mean():.1f}%")